# Model Training Pipeline

Train a machine learning model in Databricks and register it to AzureML Model Registry.

## Workflow:
1. Load feature data from Unity Catalog
2. Train model locally and with Spark MLlib
3. Evaluate model performance
4. Register model in MLflow
5. Push model to AzureML

In [ ]:
import mlflow
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib
import json
import os
import logging

# Latest Databricks SDK (databricks-sdk)
try:
    from databricks.sdk import WorkspaceClient
    print("✓ Databricks SDK (databricks-sdk) imported successfully")
except ImportError:
    print("Note: databricks-sdk not installed. Install with: pip install databricks-sdk")
    WorkspaceClient = None

# Latest Azure ML SDK v2
try:
    from azure.ai.ml import MLClient
    from azure.identity import DefaultAzureCredential, EnvironmentCredential, ChainedTokenCredential
    print("✓ Azure ML SDK v2 (azure-ai-ml) imported successfully")
except ImportError:
    print("Note: azure-ai-ml not installed. Install with: pip install azure-ai-ml")
    MLClient = None

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Set up MLflow experiment
EXPERIMENT_NAME = "/Shared/databricks-azureml-integration"
MODEL_NAME = "customer-churn-model"

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Experiment: {EXPERIMENT_NAME}")

## Step 1: Load Feature Data

In [ ]:
# Load training data from Unity Catalog
catalog = "main"
schema = "features"
table = "customer_features"

df = spark.table(f"{catalog}.{schema}.{table}")

# Convert to Pandas for sklearn
pdf = df.toPandas()

print(f"Data shape: {pdf.shape}")
print(f"Columns: {list(pdf.columns)}")
print(f"\nData types:\n{pdf.dtypes}")

## Step 2: Prepare Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Define features and target
feature_cols = [
    "amount", "hour", "day_of_week", "is_weekend", "avg_amount_7d"
]
target_col = "target_label"

# Prepare data
X = pdf[feature_cols].fillna(0)
y = pdf[target_col]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train set: {X_train_scaled.shape}")
print(f"Test set: {X_test_scaled.shape}")
print(f"Class distribution:\n{y_train.value_counts()}")

## Step 3: Train Model

In [ ]:
# Train Random Forest model
with mlflow.start_run(run_name="random-forest-v1") as run:
    # Hyperparameters
    n_estimators = 100
    max_depth = 10
    random_state = 42
    
    # Log hyperparameters
    mlflow.log_params({
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "random_state": random_state,
        "train_size": X_train_scaled.shape[0],
        "test_size": X_test_scaled.shape[0]
    })
    
    # Train model
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=random_state,
        n_jobs=-1
    )
    model.fit(X_train_scaled, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted")
    recall = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")
    
    # Log metrics
    mlflow.log_metrics({
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    })
    
    # Feature importance
    feature_importance = {col: importance for col, importance in zip(feature_cols, model.feature_importances_)}
    mlflow.log_dict(feature_importance, "feature_importance.json")
    
    # Log model
    mlflow.sklearn.log_model(model, "model", input_example=X_test_scaled[:1])
    
    run_id = run.info.run_id
    
    print(f"Run ID: {run_id}")
    print(f"\nModel Performance:")
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"\nFeature Importance:")
    for col, importance in sorted(feature_importance.items(), key=lambda x: x[1], reverse=True):
        print(f"  {col}: {importance:.4f}")

## Step 4: Register Model in MLflow

In [ ]:
# Register model in MLflow
model_uri = f"runs:/{run_id}/model"
registered_model = mlflow.register_model(model_uri, MODEL_NAME)

print(f"Model registered as: {registered_model.name}")
print(f"Version: {registered_model.version}")

# Transition to "Production" stage
client = mlflow.tracking.MlflowClient()
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=registered_model.version,
    stage="Production"
)

print(f"\nModel transitioned to Production")

## Step 5: Export Model for AzureML

In [ ]:
import tempfile
import shutil

# Create model artifacts directory
model_export_dir = f"/dbfs/dbx_demos/models/{MODEL_NAME}/v{registered_model.version}"
dbutils.fs.mkdirs(model_export_dir)

# Save model locally
model_path = f"/tmp/{MODEL_NAME}_model.pkl"
joblib.dump(model, model_path)

# Save preprocessing objects
scaler_path = f"/tmp/{MODEL_NAME}_scaler.pkl"
joblib.dump(scaler, scaler_path)

# Save metadata
metadata = {
    "model_name": MODEL_NAME,
    "version": str(registered_model.version),
    "feature_columns": feature_cols,
    "metrics": {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    },
    "feature_importance": feature_importance
}

metadata_path = f"/tmp/{MODEL_NAME}_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

# Copy to DBFS (Azure Storage)
dbutils.fs.cp(f"file:{model_path}", f"{model_export_dir}/model.pkl")
dbutils.fs.cp(f"file:{scaler_path}", f"{model_export_dir}/scaler.pkl")
dbutils.fs.cp(f"file:{metadata_path}", f"{model_export_dir}/metadata.json")

print(f"Model exported to: {model_export_dir}")
print(f"\nFiles:")
for file in dbutils.fs.ls(model_export_dir):
    print(f"  - {file.name}")

## Step 6: Prepare for AzureML Registration

In [ ]:
# Environment variables for AzureML registration
azureml_config = {
    "subscription_id": os.getenv("AZURE_SUBSCRIPTION_ID"),
    "resource_group": os.getenv("AZURE_RESOURCE_GROUP"),
    "workspace_name": os.getenv("AZUREML_WORKSPACE_NAME"),
    "model_name": MODEL_NAME,
    "model_version": str(registered_model.version),
    "model_path": model_export_dir,
    "databricks_model_uri": model_uri
}

config_path = "/tmp/azureml_config.json"
with open(config_path, "w") as f:
    json.dump(azureml_config, f, indent=2)

print("AzureML Registration Config:")
for key, value in azureml_config.items():
    if "secret" not in key.lower() and "password" not in key.lower():
        print(f"  {key}: {value}")